# Разбиение train / test

Квантильное разбиение и проверка распределений массы. Готовые индексы находятся в `data/splits/raw/` и `data/splits/augmented/`. В исходном исследовательском коде переменная `y` не задана: перед пересозданием индексов выберите для неё массив меток. Аугментированное разбиение выполняется по изображениям, без группировки копий одного скопления.

Пути определяются автоматически от корня репозитория. См. [воспроизведение](../../docs/reproduction.md) и [описание результатов](../../docs/results.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
(PROJECT_ROOT / "data/raw").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/processed/images").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/processed/images_augm").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/splits/raw").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/splits/augmented").mkdir(parents=True, exist_ok=True)

## Импорты


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap
import pandas as pd
import os

## Целевые массивы


In [ ]:
y_ = np.load(PROJECT_ROOT / "data/processed/logM500c.npy")
y_ = np.load(PROJECT_ROOT / "data/processed/logM500cUncorr.npy")

## Каталог скоплений


In [ ]:
dir_path = str(PROJECT_ROOT / "data/raw") + "/"
df = pd.read_csv(dir_path + "clusters_selected.csv")

## Квантильное разбиение


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split


def proportional_split_indices(y, test_size=0.2, n_quantiles=10, random_state=None):
    """
    Делит индексы train/test, сохраняя общее распределение y,
    даже если классы слишком редкие (без строгой стратификации).
    """
    y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    n = len(y)

    # Разбиваем по квантилям
    quantiles = np.quantile(y, np.linspace(0, 1, n_quantiles + 1))
    y_strata = np.digitize(y, quantiles, right=True)

    train_idx = []
    test_idx = []

    for q in np.unique(y_strata):
        idxs = np.where(y_strata == q)[0]
        if len(idxs) < 2:
            train_idx.extend(idxs)
            continue
        n_test = max(1, int(len(idxs) * test_size))
        test_sel = rng.choice(idxs, size=n_test, replace=False)
        train_sel = np.setdiff1d(idxs, test_sel)
        train_idx.extend(train_sel)
        test_idx.extend(test_sel)

    return np.array(train_idx), np.array(test_idx)

## Исходные данные · размеры и распределения


In [ ]:
train_idx, test_idx = proportional_split_indices(
    y, test_size=0.2, n_quantiles=2, random_state=42
)

print("Размер train:", len(train_idx))
print("Размер test:", len(test_idx))
print("Пример диапазонов:")
print("Train min/max:", y[train_idx].min(), y[train_idx].max())
print("Test min/max:", y[test_idx].min(), y[test_idx].max())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    y[train_idx], bins=30, alpha=0.6, label="Train", color="tab:blue", density=True
)
plt.hist(
    y[test_idx], bins=30, alpha=0.6, label="Test", color="tab:orange", density=True
)
plt.xlabel("log10(M500cUncorr)")
plt.ylabel("Density")
plt.title("Train / test log-mass distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    y_[train_idx], bins=30, alpha=0.6, label="Train", color="tab:blue", density=True
)
plt.hist(
    y_[test_idx], bins=30, alpha=0.6, label="Test", color="tab:orange", density=True
)
plt.xlabel("log10(M500cUncorr)")
plt.ylabel("Density")
plt.title("Train / test log-mass distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Сохранение индексов исходной выборки


In [ ]:
np.save(PROJECT_ROOT / "data/splits/raw/train_indices.npy", train_idx)
np.save(PROJECT_ROOT / "data/splits/raw/test_indices.npy", test_idx)

## Расширенная выборка


In [ ]:
y_augm = np.load(PROJECT_ROOT / "data/processed/logM500cUncorr_augm.npy")

In [ ]:
y_augm.shape

## Разбиение и распределение после аугментации


In [ ]:
train_idx, test_idx = proportional_split_indices(
    y_augm, test_size=0.2, n_quantiles=2, random_state=42
)

print("Размер train:", len(train_idx))
print("Размер test:", len(test_idx))
print("Пример диапазонов:")
print("Train min/max:", y_augm[train_idx].min(), y_augm[train_idx].max())
print("Test min/max:", y_augm[test_idx].min(), y_augm[test_idx].max())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    y_augm[train_idx], bins=30, alpha=0.6, label="Train", color="tab:blue", density=True
)
plt.hist(
    y_augm[test_idx], bins=30, alpha=0.6, label="Test", color="tab:orange", density=True
)
plt.xlabel("log10(M500cUncorr)")
plt.ylabel("Density")
plt.title("Train / test log-mass distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Сохранение индексов расширенной выборки


In [ ]:
np.save(PROJECT_ROOT / "data/splits/augmented/train_indices.npy", train_idx)
np.save(PROJECT_ROOT / "data/splits/augmented/test_indices.npy", test_idx)